<a href="https://colab.research.google.com/github/Noors-lab/toll-plaza-vision-system/blob/main/data_for_YOLOV8" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



connecting to my google drive



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


checking if the video files are in right folder or not

In [ ]:
import os
files = os.listdir('/content/drive/MyDrive/vision_data/')
print(files)


['day_cctv_0:00:41.mp4', 'night_highway_dashcam_footage_3:54:54.mp4', 'small_day_cctv_footage_0:00:24.mp4', 'traffic_night_cctv_footage_1:09:02.mp4', 'rain_fog_dashcam_footage_8:02:55.mp4', 'validation_image', 'training_day_image', 'extracted_frames']


# the frame extractor cell

In [ ]:
import cv2
import os

VIDEOS = [
    {
        "path": "/content/drive/MyDrive/vision_data/night_highway_dashcam_footage_3:54:54.mp4",
        "label": "night_moving",
        "fps_sample": 0.5,
    },
    {
        "path": "/content/drive/MyDrive/vision_data/rain_fog_dashcam_footage_8:02:55.mp4",
        "label": "rain_fog_moving",
        "fps_sample": 0.3,
    },
    {
        "path": "/content/drive/MyDrive/vision_data/traffic_night_cctv_footage_1:09:02.mp4",
        "label": "night_cctv",
        "fps_sample": 0.5,
    },
    {
        "path": "/content/drive/MyDrive/vision_data/day_cctv_0:00:41.mp4",
        "label": "day_cctv",
        "fps_sample": 2.0,
    },
    {
        "path": "/content/drive/MyDrive/vision_data/small_day_cctv_footage_0:00:24.mp4",
        "label": "day_cctv_small",
        "fps_sample": 2.0,
    },
]

OUTPUT_ROOT = "/content/drive/MyDrive/vision_data/extracted_frames"

def extract(video_path, label, fps_sample):
    out_dir = os.path.join(OUTPUT_ROOT, label)
    os.makedirs(out_dir, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"  [ERROR] Cannot open: {video_path}")
        return

    native_fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration_min = total_frames / native_fps / 60
    interval = max(1, int(native_fps / fps_sample))

    print(f"\n[{label}]")
    print(f"  Duration: {duration_min:.1f} min  |  Native FPS: {native_fps:.1f}")
    print(f"  Saving 1 frame every {interval} frames")

    saved = 0
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % interval == 0:
            filename = f"{label}_{saved:05d}.jpg"
            cv2.imwrite(os.path.join(out_dir, filename), frame, [cv2.IMWRITE_JPEG_QUALITY, 90])
            saved += 1
        frame_idx += 1

    cap.release()
    print(f"  Saved: {saved} frames → {out_dir}/")

os.makedirs(OUTPUT_ROOT, exist_ok=True)
for v in VIDEOS:
    extract(v["path"], v["label"], v["fps_sample"])

print("\nDONE!")

total = sum(len(files) for _, _, files in os.walk(OUTPUT_ROOT))
print(f"Total frames extracted: {total}")


[night_moving]
  Duration: 234.9 min  |  Native FPS: 24.0
  Saving 1 frame every 47 frames


KeyboardInterrupt: 

# checking the structure of the extracted frames

In [ ]:
import os
OUTPUT_ROOT = "/content/drive/MyDrive/vision_data/extracted_frames"
for folder in os.listdir(OUTPUT_ROOT):
    folder_path = os.path.join(OUTPUT_ROOT, folder)
    count = len(os.listdir(folder_path))
    print(f"{folder}: {count} frames")

night_moving: 7190 frames
rain_fog_moving: 8772 frames
night_cctv: 2072 frames
day_cctv: 84 frames
day_cctv_small: 52 frames


balancing frames to fit in roboflow's free tier account

In [ ]:
import os, shutil, random

SOURCE = "/content/drive/MyDrive/vision_data/extracted_frames"
DEST = "/content/drive/MyDrive/vision_data/sampled_frames"

SAMPLE = {
    "night_moving": 1000,
    "rain_fog_moving": 1000,
    "night_cctv": 800,
    "day_cctv": 84,
    "day_cctv_small": 52,
}

for folder, n in SAMPLE.items():
    src = os.path.join(SOURCE, folder)
    dst = os.path.join(DEST, folder)
    os.makedirs(dst, exist_ok=True)
    files = os.listdir(src)
    sampled = random.sample(files, min(n, len(files)))
    for f in sampled:
        shutil.copy(os.path.join(src, f), os.path.join(dst, f))
    print(f"{folder}: {len(sampled)} frames sampled")

print("\nDONE!")

night_moving: 1000 frames sampled
rain_fog_moving: 1000 frames sampled
night_cctv: 800 frames sampled
day_cctv: 84 frames sampled
day_cctv_small: 52 frames sampled

DONE!


# sampled folder path

In [ ]:
import os
DEST = "/content/drive/MyDrive/vision_data/sampled_frames"
for folder in os.listdir(DEST):
    print(folder)


night_moving
rain_fog_moving
night_cctv
day_cctv
day_cctv_small
